<a href="https://colab.research.google.com/github/joega66/next-gen-photogrammetry/blob/main/notebook28a537cec6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Dependency: Immediate 3DGS (`i3dgs`)

[graphdeco-inria/i3dgs](https://github.com/graphdeco-inria/i3dgs) — *Immediate 3D Gaussian Splat Reconstruction of Unordered Input with Global Consistency* (SIGGRAPH '26). It goes straight from a folder of unordered images to a Gaussian splat scene: no COLMAP pass required.

**Before running the setup cell:**

- An **NVIDIA GPU is required**. `submodules/diff-gaussian-rasterization` and `submodules/simple-knn` are CUDA extensions compiled from source on first install — budget ~10–20 min.
- The repo pins **torch 2.7.1 / cu128**. The setup cell reinstalls torch to match, so it must run **before anything imports torch** in this kernel. If you have already imported torch, restart the runtime first.
- Reconstruction reads images from `${SOURCE_PATH}/images` (`.png`, `.jpg`, `.jpeg`, `.webp`, consumed in alphabetical order). `${SOURCE_PATH}/sparse/0` COLMAP output is optional and only used for pose visualization.

### Why the install is split into three phases

The README's one-liner (`pip install -r requirements.txt --no-build-isolation`) **cannot work on Colab or Kaggle**, which run Python 3.13:

1. `depth-anything-3` declares `numpy<2`. The last numpy 1.x release (1.26.4) ships no cp313 wheels, so pip must build it from source.
2. `--no-build-isolation` applies to the whole file, so numpy's `meson-python` build backend is never installed → `ModuleNotFoundError: No module named 'mesonpy'`.

The setup cell therefore installs the ordinary wheels normally, installs DA3 with `--no-deps` plus only the packages `depth_anything_3.api` actually imports, and reserves `--no-build-isolation` for the three torch-linked builds that genuinely need it.

> ⚠️ **This overrides an upstream pin.** DA3 asks for `numpy<2` and it gets numpy 2.x. Its source uses no numpy-2-removed APIs (`np.float_`, `np.int_`, `np.bool8` are all absent), so this should be safe — but it is untested by upstream. If you hit a numpy dtype error inside `depth_anything_3`, that pin is the first suspect. The alternative is a Python 3.12 environment, where `numpy<2` resolves to a wheel and the README one-liner works as written.


In [ ]:
# --- i3dgs: clone + install ------------------------------------------------
# Run this BEFORE importing torch anywhere in the kernel (it pins torch 2.7.1).
#
# NOTE: this deliberately does NOT run the README's single
#   pip install -r requirements.txt --no-build-isolation
# On a Python 3.13 host (Colab/Kaggle) that command cannot succeed — see the
# phase split and the numpy note below.
import os, shutil, subprocess, sys
from pathlib import Path

I3DGS_URL = "https://github.com/graphdeco-inria/i3dgs.git"
I3DGS_DIR = Path("/kaggle/working/i3dgs") if Path("/kaggle/working").is_dir() else Path.cwd() / "i3dgs"
DA3_URL = "git+https://github.com/ByteDance-Seed/Depth-Anything-3.git"
DA3_PIN = "40674546e40144452ec93035ac77d762f4b5d16f"  # sha from i3dgs requirements.txt
PIN_TORCH = True      # repo targets torch 2.7.1 + torchvision 0.22.1
CUDA_WHEEL = "cu128"  # upstream also publishes cu118 and rocm6.3

PIP = f"{sys.executable} -m pip install --progress-bar off"


def sh(cmd, cwd=None, quiet=True):
    """Run a command, streaming or buffering output, but ALWAYS showing it on failure."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=None if cwd is None else str(cwd),
                            text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    captured = []
    for line in proc.stdout:
        captured.append(line)
        if not quiet:
            print(line, end="", flush=True)
    if proc.wait():
        if quiet:  # never let a failure go unexplained
            print("".join(captured), flush=True)
        raise RuntimeError(f"exit {proc.returncode}: {cmd}")


if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU visible — i3dgs needs CUDA. Switch the runtime to a GPU accelerator.")

# Target only this machine's compute capability so the CUDA extensions build once, quickly.
compute_cap = subprocess.run(
    "nvidia-smi --query-gpu=compute_cap --format=csv,noheader",
    shell=True, capture_output=True, text=True, check=True,
).stdout.splitlines()[0].strip()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{compute_cap}+PTX"
print("building for compute capability", compute_cap, "| python", sys.version.split()[0])

# --recursive: diff-gaussian-rasterization and simple-knn are submodules.
if not I3DGS_DIR.exists():
    sh(f"git clone --recursive {I3DGS_URL} {I3DGS_DIR}")
else:
    sh("git submodule update --init --recursive", cwd=I3DGS_DIR)

# hatch-vcs: DA3 sets [tool.hatch.version] source = "vcs", so the build backend
# needs the plugin present to resolve its own version.
sh(f"{PIP} hatchling hatch-vcs ninja")
if PIN_TORCH:
    sh(f"{PIP} torch==2.7.1 torchvision==0.22.1 "
       f"--index-url https://download.pytorch.org/whl/{CUDA_WHEEL}")

# Guard rail: nothing below may drag numpy back to 1.x. numpy 1.26.4 is the last
# 1.x release and ships no cp313 wheels, so a downgrade means building it from
# source — which is what produced the "No module named 'mesonpy'" failure.
constraints = I3DGS_DIR / "nb-constraints.txt"
constraints.write_text("numpy>=2.1,<3\n")

# -- phase 1: plain wheels, WITH build isolation ----------------------------
sh(f"{PIP} -c {constraints} meshio tqdm opencv-python lpips addict kornia joblib timm")

# -- phase 2: Depth-Anything-3 ----------------------------------------------
# --no-deps: DA3 declares numpy<2, unsatisfiable as a wheel on Python 3.13. Its
# source uses no removed-in-numpy-2 APIs, so we supply the deps that
# `depth_anything_3.api` actually imports and leave numpy at 2.x. Skipped on
# purpose: xformers, open3d, pycolmap, fastapi/uvicorn/typer, e3nn, pre-commit.
#
# --no-build-isolation: DA3's version comes from hatch-vcs, which inspects git
# tags. pip clones the pinned sha detached and tagless, so an isolated build
# cannot derive a version. Building against this env's hatchling avoids that.
sh(f"{PIP} --no-deps --no-build-isolation {DA3_URL}@{DA3_PIN}")
sh(f"{PIP} -c {constraints} huggingface_hub pillow omegaconf addict einops "
   f"imageio plyfile matplotlib safetensors evo moviepy==1.0.3")

# -- phase 3: torch-linked builds, WITHOUT build isolation ------------------
# These compile against the torch installed above, so they must see it at build
# time. The CUDA extensions take the longest — stream their output.
sh(f"{PIP} --no-build-isolation -c {constraints} git+https://github.com/rahul-goel/fused-ssim")
sh(f"{PIP} --no-build-isolation -c {constraints} "
   f"graphdecoviewer@git+https://github.com/graphdeco-inria/graphdecoviewer@i3dgs-fixes")
print("\ncompiling CUDA extensions — this is the slow part (~10-20 min):")
sh(f"{PIP} --no-build-isolation -c {constraints} "
   f"./submodules/diff-gaussian-rasterization ./submodules/simple-knn",
   cwd=I3DGS_DIR, quiet=False)
sh(f"{PIP} -c {constraints} cupy-cuda12x")

print("\ni3dgs installed at", I3DGS_DIR)

In [ ]:
# --- i3dgs: verify the install ---------------------------------------------
# Fails loudly here rather than deep inside the first reconstruction.
import importlib

if str(I3DGS_DIR) not in sys.path:
    sys.path.insert(0, str(I3DGS_DIR))

REQUIRED = [
    ("diff_gaussian_rasterization", "CUDA rasterizer (submodule)"),
    ("simple_knn._C",               "KNN (submodule)"),
    ("fused_ssim",                  "SSIM loss"),
    ("cupy",                        "RANSAC / bundle-adjustment kernels"),
    ("depth_anything_3.api",        "pose init — the --no-deps install lands here"),
    ("graphdecoviewer",             "imported by train.py even when viewer_mode='none'"),
    ("imgui_bundle",                "imported by gaussianviewer, which train.py imports"),
    ("kornia", ""), ("lpips", ""), ("meshio", ""), ("timm", ""), ("cv2", ""),
]

failed = []
for mod, why in REQUIRED:
    try:
        importlib.import_module(mod)
        print(f"  ok    {mod}")
    except Exception as exc:
        failed.append(mod)
        print(f"  FAIL  {mod:30} {type(exc).__name__}: {exc}" + (f"  [{why}]" if why else ""))

import numpy
print(f"\nnumpy {numpy.__version__} — must be 2.x; a 1.x downgrade means no cp313 wheels")
if failed:
    raise ImportError("install incomplete: " + ", ".join(failed))
print("all i3dgs imports resolved")

In [ ]:
# --- i3dgs: import the reconstruction pipeline -----------------------------
import contextlib, random
import numpy as np
import torch

if str(I3DGS_DIR) not in sys.path:
    sys.path.insert(0, str(I3DGS_DIR))
# get_args() derives an unset --model_path as results/NNNNNN relative to cwd,
# and the repo resolves its checkpoints/assets relative to the repo root too.
os.chdir(I3DGS_DIR)

from args import get_args            # noqa: E402
from train import ReconstructionTask  # noqa: E402

print("torch", torch.__version__, "| cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0))


@contextlib.contextmanager
def _argv(argv):
    saved, sys.argv = sys.argv, argv
    try:
        yield
    finally:
        sys.argv = saved


def run_i3dgs(source_path, model_path=None, num_iterations=30, test_hold=-1,
              viewer_mode="none", extra_args=()):
    """Run the immediate-3DGS reconstruction on ``${source_path}/images``.

    ``extra_args`` is a list of raw CLI flags passed through to ``args.get_args``
    (e.g. ``["--test_frequency", "10"]``). Returns the model output directory.
    """
    source_path = Path(source_path)
    if not (source_path / "images").is_dir():
        raise FileNotFoundError(f"{source_path}/images does not exist — i3dgs reads its frames from there.")

    argv = ["train.py",
            "-s", str(source_path),
            "--num_iterations", str(num_iterations),
            "--test_hold", str(test_hold),
            "--viewer_mode", viewer_mode]
    if model_path is not None:
        argv += ["-m", str(model_path)]
    argv += [str(a) for a in extra_args]

    # get_args() parses sys.argv directly, so hand it an argv rather than a kwargs dict.
    with _argv(argv):
        args = get_args()

    # Mirrors train.py's __main__ block, which we bypass by calling the task directly.
    torch.random.manual_seed(0)
    torch.cuda.manual_seed(0)
    np.random.seed(0)
    random.seed(0)
    if args.deterministic_poses:
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.use_deterministic_algorithms(True, warn_only=True)
        args.lr_poses = 0.0

    ReconstructionTask(args).run()
    return Path(args.model_path)

In [ ]:
# --- i3dgs: choose the input capture ---------------------------------------
# Point SOURCE_PATH at a folder containing an `images/` subdirectory. On Kaggle
# that is typically an attached dataset under /kaggle/input/<slug>.
SOURCE_PATH = Path("/kaggle/input/CHANGE-ME")

# Or grab one of the paper's scenes to smoke-test the pipeline:
# sh(f"{sys.executable} scripts/download_datasets.py --out_dir data/", cwd=I3DGS_DIR)
# SOURCE_PATH = I3DGS_DIR / "data" / "<scene>"

IMG_EXT = {".png", ".jpg", ".jpeg", ".webp"}
frames = sorted(p for p in (SOURCE_PATH / "images").iterdir() if p.suffix.lower() in IMG_EXT) \
    if (SOURCE_PATH / "images").is_dir() else []
print(f"{SOURCE_PATH}: {len(frames)} frames")
if frames:
    print("first:", frames[0].name, "| last:", frames[-1].name)
else:
    print(f"!! no frames found — expected images under {SOURCE_PATH / 'images'}")

In [ ]:
# --- i3dgs: run the reconstruction -----------------------------------------
MODEL_PATH = Path("/kaggle/working/i3dgs_out") if Path("/kaggle/working").is_dir() \
    else I3DGS_DIR / "results" / "notebook"

model_dir = run_i3dgs(
    SOURCE_PATH,
    model_path=MODEL_PATH,
    num_iterations=30,   # optimization iterations per keyframe
    test_hold=-1,        # e.g. 8 to hold out every 8th image for eval metrics
    viewer_mode="none",  # no interactive viewer in a notebook kernel
)
print("reconstruction written to", model_dir)

In [ ]:
# --- i3dgs: inspect the outputs --------------------------------------------
for p in sorted(model_dir.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:9.2f} MB  {p.relative_to(model_dir)}")

# Render a fly-through video from a saved camera path (see the repo's README for
# the render-path format); the viewer itself needs a display, so it is CLI-only:
#   !python scripts/render_path.py -m {model_dir} --render_path <PATH> --out_dir <VIDEO_DIR>
#   !python gaussianviewer.py local {model_dir}